In [11]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd
import time
import re

def bygg_url(trotter_id_eller_navn):
    if str(trotter_id_eller_navn).isdigit():
        return f"https://www.blodbanken.nu/servlet/GetBBData?trotter={trotter_id_eller_navn}"
    else:
        trotter_navn_encoded = trotter_id_eller_navn.replace(" ", "+")
        # Use 'cold' for coldblood trotters or 'warm' for warmbloods (SE horses are typically warmbloods)
        return f"https://www.blodbanken.nu/servlet/GetBBData?breed=warm&trotter={trotter_navn_encoded}"

def get_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=chrome_options)
    return driver

def hent_navn_fra_existing_horse(soup):
    try:
        td_elements = soup.find_all('td')
        for td in td_elements:
            if 'Existing horse' in td.get_text():
                next_td = td.find_next_sibling('td')
                if next_td:
                    horse_name = next_td.get_text(strip=True)
                    if horse_name and horse_name != '>>':
                        print(f"Navn funnet: {horse_name}")
                        return horse_name
        
        for element in soup.find_all(['b', 'strong']):
            text = element.get_text(strip=True)
            pattern = r'^[A-Za-z][A-Za-z\s\'\-\.]+\([A-Z]{2}\)$'
            if re.match(pattern, text):
                print(f"Navn funnet: {text}")
                return text
        
        for td in soup.find_all('td'):
            text = td.get_text(strip=True)
            pattern = r'^([A-Za-z][A-Za-z\s\'\-\.]+\([A-Z]{2}\))'
            match = re.match(pattern, text)
            if match:
                name = match.group(1)
                print(f"Navn funnet: {name}")
                return name
    except Exception as e:
        print(f"Feil ved henting av navn: {e}")
    
    print("Navn ikke funnet")
    return None

def extract_blup_values(soup):
    blup_data = {
        "Number of starts": None,
        "Racing Performance": None,
        "Percentage of starters": None,
        "Ancestry index": None,
        "Dev": None,
        "Total index": None,
        "Accuracy": None
    }
    
    try:
        all_rows = soup.find_all("tr")
        for row in all_rows:
            cols = row.find_all("td")
            if len(cols) >= 2:
                label_text = cols[0].get_text(strip=True)
                value_text = cols[1].get_text(strip=True)
                if "Number of starts" in label_text:
                    blup_data["Number of starts"] = value_text
                elif "Racing Performance" in label_text:
                    blup_data["Racing Performance"] = value_text
                elif "Percentage of starters" in label_text:
                    blup_data["Percentage of starters"] = value_text
                elif "Ancestry index" in label_text:
                    blup_data["Ancestry index"] = value_text
                elif label_text == "Dev":
                    blup_data["Dev"] = value_text
                elif "Total index" in label_text:
                    blup_data["Total index"] = value_text
                elif "Accuracy" in label_text:
                    blup_data["Accuracy"] = value_text
        found_values = sum(1 for v in blup_data.values() if v is not None)
        if found_values > 0:
            print(f"BLUP data extracted: {found_values}/7 fields")
        else:
            print("No BLUP values found")
    except Exception as e:
        print(f"Error extracting BLUP values: {e}")
    return blup_data

def extract_inbreeding_coefficient(soup):
    stc_value = None
    blood_bank_value = None
    
    try:
        all_rows = soup.find_all("tr")
        
        for row in all_rows:
            cols = row.find_all("td")
            if len(cols) >= 2:
                label_text = cols[0].get_text(strip=True)
                value_text = cols[1].get_text(strip=True)
                
                # Look for STC coefficient
                if label_text == "Inbreeding Coefficient (STC)":
                    pattern = r'(\d+,\d+)\s*%'
                    match = re.search(pattern, value_text)
                    if match:
                        stc_value = match.group(1) + " %"
                        print(f"Inbreeding Coefficient (STC) extracted: {stc_value}")
                    elif 'Not available' in value_text or 'not available' in value_text.lower():
                        stc_value = "Not available"
                        print("Inbreeding Coefficient (STC): Not available")
                    else:
                        pattern2 = r'(\d+\.\d+)\s*%'
                        match2 = re.search(pattern2, value_text)
                        if match2:
                            stc_value = match2.group(1).replace('.', ',') + " %"
                            print(f"Inbreeding Coefficient (STC) extracted: {stc_value}")
                        else:
                            stc_value = value_text
                
                # Look for Blood Bank coefficient
                if "Inbreeding Coefficient (The Blood Bank" in label_text:
                    pattern = r'(\d+,\d+)\s*%'
                    match = re.search(pattern, value_text)
                    if match:
                        blood_bank_value = match.group(1) + " %"
                        print(f"Inbreeding Coefficient (The Blood Bank) extracted: {blood_bank_value}")
                    else:
                        pattern2 = r'(\d+\.\d+)\s*%'
                        match2 = re.search(pattern2, value_text)
                        if match2:
                            blood_bank_value = match2.group(1).replace('.', ',') + " %"
                            print(f"Inbreeding Coefficient (The Blood Bank) extracted: {blood_bank_value}")
        
        # Return both values as a tuple
        if not stc_value:
            print("Inbreeding Coefficient (STC) not found")
        if not blood_bank_value:
            print("Inbreeding Coefficient (The Blood Bank) not found")
            
        return stc_value, blood_bank_value
        
    except Exception as e:
        print(f"Error extracting Inbreeding Coefficient: {e}")
        return None, None

def hent_blodlinjer(trotter_id_eller_navn):
    url = bygg_url(trotter_id_eller_navn)
    driver = get_driver()
    driver.get(url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    # Check if we're on a search results page with multiple horses
    search_results = soup.find_all('a', href=lambda x: x and 'trotter=' in str(x))
    se_horse_link = None
    
    # Look for a horse with (SE) in the link text
    for link in search_results:
        link_text = link.get_text(strip=True)
        if '(SE)' in link_text and 'Existing horse' not in link_text:
            se_horse_link = link.get('href')
            print(f"Found SE horse in search results: {link_text}")
            break
    
    # If we found an SE horse, navigate to that page
    if se_horse_link:
        if not se_horse_link.startswith('http'):
            se_horse_link = 'https://www.blodbanken.nu' + se_horse_link
        print(f"Navigating to SE horse: {se_horse_link}")
        driver.get(se_horse_link)
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")
    
    driver.quit()
    
    data = {
        "ID/Navn": trotter_id_eller_navn,
        "Navn": None,
        "French Trotter": None,
        "Russian Trotter": None,
        "Standardbred": None,
        "Generation interval (average, 4 gen)": None,
        "Inbreeding Coefficient (STC)": None,
        "Inbreeding Coefficient (The Blood Bank)": None
    }
    
    data["Navn"] = hent_navn_fra_existing_horse(soup)
    
    for table in soup.find_all("table"):
        if "Breeds" in table.get_text():
            rows = table.find_all("tr")
            for row in rows:
                cols = row.find_all("td")
                if len(cols) == 2:
                    rase = cols[0].get_text(strip=True)
                    andel = cols[1].get_text(strip=True).replace("‰", "").strip()
                    if rase in ["French Trotter", "Russian Trotter", "Standardbred"]:
                        data[rase] = andel
            break
    
    try:
        gen_element = soup.find("td", string=lambda x: x and "Generation interval (average, 4 gen)" in x)
        if gen_element:
            gen_value_td = gen_element.find_next_sibling("td")
            if gen_value_td:
                raw_text = gen_value_td.get_text(separator=" ", strip=True)
                pattern = r"(\d{1,2}[,.]?\d{1,2})"
                match = re.search(pattern, raw_text)
                if match:
                    data["Generation interval (average, 4 gen)"] = match.group(1)
                else:
                    data["Generation interval (average, 4 gen)"] = "Not available"
    except Exception as e:
        print(f"Klarte ikke hente generation interval: {e}")
    
    stc_coefficient, blood_bank_coefficient = extract_inbreeding_coefficient(soup)
    data["Inbreeding Coefficient (STC)"] = stc_coefficient
    data["Inbreeding Coefficient (The Blood Bank)"] = blood_bank_coefficient
    
    blup_data = extract_blup_values(soup)
    data.update(blup_data)
    
    return data

def hent_blodlinjer_for_liste(trotter_liste):
    resultater = []
    for trotter in trotter_liste:
        print(f"\nHenter data for: {trotter}")
        bloddata = hent_blodlinjer(trotter)
        resultater.append(bloddata)
    df = pd.DataFrame(resultater)
    return df

if __name__ == "__main__":
    trotter_liste = [
    "Tess Tilly",
    "Thorn Bird",
    "To Russia",
    "Touch of Elegance",
    "Unison Sund",
    "Valetta Strix",
    "Venezia As",
    "Victoria Empress",
    "What's Your Limit",
    "Wink Spot",
    "Woozy",
    "Yquem Wine",
    "Zara Barosso",
    "Albright",
    "All April",
    "Amazing Sensation",
    "Anguilla River",
    "Avrilvingtun Cam",
    "Bank Wise As",
    "Billie Claire",
    "Blue Martini Ås",
    "Southwind Cartier",
    "Chapter Review",
    "Crazy in Lindy",
    "Crone",
    "Dance Beat",
    "Dancin in the Rain",
    "Dimanche Mearas",
    "Dismoiouicheri",
    "Eileen Boko",
    "Enjoy de Luxe",
    "Explosive Rain",
    "Fair Enough Am",
    "Faloria Boko",
    "Gazelle Am",
    "Girlpower Pop",
    "Gloria Sisu",
    "Mellby Happy",
    "Helloise Boko",
    "Horoscope",
    "Hotentrot Hanover",
    "Icone Am",
    "Illustre November",
    "Indiana Boko",
    "Ioana Hanover",
    "Jaramaz Boko",
    "Katla Celeber",
    "Kayster Boko",
    "Keyla Wibb",
    "Kissmeonceagain",
    "Kroatia Boko",
    "Make Approach",
    "Mandatory",
    "Meryem K.Boko",
    "Lima Midnite Magic",
    "Mindyourvoice W.F.",
    "Miracle Tile",
    "Moviestar",
    "Musique d'Inverne",
    "My Way I.H.",
    "Mystery Tile",
    "Nancy America",
    "Nephenta Lux",
    "No Business",
    "Overtop Kronos",
    "Passion Tile",
    "Patricks Princess",
    "Quick Emotion",
    "Rania",
    "Rare Wise As",
    "Västerbo Realworld",
    "Rugiada Dei Rex",
    "Sansuna Hanover",
    "Skat Face",
    "Somebodylikeyou Ås",
    "Spicy Chocolate",
    "Lima Star",
    "Starbritestarlite",
    "System Performance"
]
    blodlinjer_df = hent_blodlinjer_for_liste(trotter_liste)
    print("\n" + "="*80)
    print("RESULTATER:")
    print("="*80)
    print(blodlinjer_df)
    output_path = "blodbank_data_with_blup.xlsx"
    blodlinjer_df.to_excel(output_path, index=False)
    print(f"\nData lagret til: {output_path}")
    csv_output_path = "blodbank_data_with_blup.csv"
    blodlinjer_df.to_csv(csv_output_path, index=False)
    print(f"Data ogsa lagret til: {csv_output_path}")


Henter data for: Tess Tilly
Navn funnet: Tess Tilly (SE)
Inbreeding Coefficient (The Blood Bank) extracted: 31,62 %
Inbreeding Coefficient (The Blood Bank) extracted: 6,878 %
Inbreeding Coefficient (STC) extracted: 6,260 %
BLUP data extracted: 7/7 fields

Henter data for: Thorn Bird
Navn funnet: Thorn Bird (SE)
Inbreeding Coefficient (The Blood Bank) extracted: 0,00 %
Inbreeding Coefficient (The Blood Bank) extracted: 15,120 %
Inbreeding Coefficient (STC) extracted: 13,920 %
BLUP data extracted: 7/7 fields

Henter data for: To Russia
Navn funnet: To Russia (US)
Inbreeding Coefficient (The Blood Bank) extracted: 0,00 %
Inbreeding Coefficient (The Blood Bank) extracted: 12,992 %
Inbreeding Coefficient (STC): Not available
No BLUP values found

Henter data for: Touch of Elegance
Navn funnet: Touch of Elegance (SE)
Inbreeding Coefficient (STC) not found
Inbreeding Coefficient (The Blood Bank) not found
No BLUP values found

Henter data for: Unison Sund
Navn funnet: Unison Sund (SE)
Inbree